# Demonstração do pipeline de pré-processamento e cálculo de compatibilidade (TF-IDF)

Exibe a **saída de cada etapa** do sistema, do texto bruto de um currículo ao *score* final de compatibilidade com uma vaga. As saídas podem ser copiadas para o relatório (Quadro/Tabela).

Etapas: normalização → tokenização → lematização → remoção de *stopwords* → TF-IDF → similaridade do cosseno → extração de habilidades → *score* de habilidades e *score* final.

In [ ]:
# Localiza a raiz do projeto (pasta que contém 'src') e a adiciona ao sys.path.
# Funciona rodando de qualquer pasta e também via 'jupyter nbconvert --execute'.
import sys, os
_raiz = os.path.abspath(os.getcwd())
while _raiz != os.path.dirname(_raiz):
    if os.path.isdir(os.path.join(_raiz, 'src')):
        break
    _raiz = os.path.dirname(_raiz)
sys.path.insert(0, _raiz)

from src.preprocessamento.limpador_texto import LimpadorTexto, limpar_texto

limpador = LimpadorTexto()
print('Módulo carregado com sucesso. Raiz do projeto:', _raiz)

In [ ]:
# Texto de exemplo com ruídos típicos de um currículo real
trecho = (
    'João Silva | joao.silva@email.com | (11) 93206-5400 | https://linkedin.com/in/joao\n'
    'Experiência com node.js, c# e asp.net. Conhecimento em ml e nlp. '
    'Responsável pela administração dos servidores e infraestrutura de redes.'
)

print('TEXTO ORIGINAL:')
print(trecho)

In [ ]:
# Etapa 1 — Remoção de URLs
t1 = limpador.remover_urls(trecho)
print('Após remover URLs:')
print(t1)

In [ ]:
# Etapa 2 — Remoção de e-mails
t2 = limpador.remover_emails(t1)
print('Após remover e-mails:')
print(t2)

In [ ]:
# Etapa 3 — Remoção de telefones
t3 = limpador.remover_telefones(t2)
print('Após remover telefones:')
print(t3)

In [ ]:
# Etapa 4 — Normalização de termos técnicos
t4 = limpador.normalizar_termos_tecnicos(t3)
print('Após normalizar termos técnicos:')
print(t4)
print()

from src.preprocessamento.limpador_texto import NORMALIZACOES
print('Exemplos de normalizações configuradas:')
exemplos = [('node.js','nodejs'),('c#','csharp'),('c++','cplusplus'),('asp.net','aspnet'),
            ('.net','dotnet'),('ml','machinelearning'),('ai','artificialintelligence'),
            ('nlp','naturallanguageprocessing'),('gcp','googlecloudplatform')]
for antes, depois in exemplos:
    print(f'  {antes:<12} -> {depois}')

In [ ]:
# Etapa 5 — Remoção de caracteres especiais
t5 = limpador.remover_caracteres_especiais(t4)
print('Após remover caracteres especiais:')
print(t5)

In [ ]:
# Etapa 6 — Remoção de acentos
t6 = limpador.remover_acentos(t5)
print('Após remover acentos:')
print(t6)

In [ ]:
# Etapa 7 — Tokenização, lematização e remoção de stopwords (spaCy)
tokens = limpador.tokenizar_e_limpar(t6)
print(f'Tokens após lematização e remoção de stopwords ({len(tokens)} tokens):')
print(tokens)
print()

from src.preprocessamento.limpador_texto import STOPWORDS_DOMINIO
palavras_t6 = t6.lower().split()
removidas = [p for p in palavras_t6 if p in STOPWORDS_DOMINIO]
print('Stopwords de domínio removidas neste texto:')
print(removidas if removidas else '(nenhuma neste trecho)')

In [ ]:
# Resumo: texto bruto vs. texto após limpeza completa
texto_limpo = limpar_texto(trecho)

print('TEXTO BRUTO:')
print(trecho)
print()
print('TEXTO APÓS LIMPEZA COMPLETA:')
print(texto_limpo)
print()
print(f'Caracteres antes : {len(trecho)}')
print(f'Caracteres depois: {len(texto_limpo)}')
print(f'Reducao          : {round((1 - len(texto_limpo)/len(trecho)) * 100, 1)}%')

In [ ]:
# Quadro consolidado das etapas de pré-processamento (copiar para o relatório)
import pandas as pd
nl = chr(10)
etapas = [
    ('0. Texto original', trecho.replace(nl,' ')),
    ('1. Sem URLs', t1.replace(nl,' ')),
    ('2. Sem e-mails', t2.replace(nl,' ')),
    ('3. Sem telefones', t3.replace(nl,' ')),
    ('4. Termos técnicos normalizados', t4.replace(nl,' ')),
    ('5. Sem caracteres especiais', t5.replace(nl,' ')),
    ('6. Sem acentos', t6.replace(nl,' ')),
    ('7. Tokens finais (lematizados, sem stopwords)', ' '.join(tokens)),
]
pd.set_option('display.max_colwidth', None)
pd.DataFrame(etapas, columns=['Etapa', 'Saída'])

In [ ]:
# Efeito da lematização: formas diferentes -> mesma raiz
import spacy
nlp = spacy.load('pt_core_news_sm')
exemplos_lema = ['desenvolvidas','desenvolvendo','desenvolveu','experiências','experiência',
                 'trabalhando','trabalhou','trabalhos','gerenciando','gerenciou','gerenciamento']
print(f'{"Palavra original":<22} -> Lema (forma base)')
print('-'*45)
for palavra in exemplos_lema:
    print(f'{palavra:<22} -> {nlp(palavra)[0].lemma_.lower()}')

## TF-IDF, similaridade do cosseno e cálculo dos scores

Comparamos um **currículo** com uma **vaga** de exemplo. As Etapas 8 e 9 ilustram o **conceito** do TF-IDF e do cosseno com um vetorizador simples; os valores **oficiais do sistema** (com os pesos reais configurados no matcher) aparecem na Etapa 11.

In [ ]:
# Documentos de exemplo: um currículo e uma vaga (texto bruto)
curriculo = ('Desenvolvedor back-end com experiência em Python, Docker e PostgreSQL. '
             'Atuação com APIs REST e integração contínua.')
vaga = ('Vaga para desenvolvedor Python. Desejável Docker, bancos de dados relacionais '
        'como PostgreSQL e experiência com APIs REST.')

curriculo_limpo = limpar_texto(curriculo)
vaga_limpa = limpar_texto(vaga)
print('Currículo (limpo):', curriculo_limpo)
print('Vaga (limpa)     :', vaga_limpa)

In [ ]:
# Etapa 8 — TF-IDF (ilustração): pesos de cada termo nos dois documentos
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer()
matriz = vec.fit_transform([curriculo_limpo, vaga_limpa])
tabela_tfidf = pd.DataFrame(matriz.toarray().T, index=vec.get_feature_names_out(),
                            columns=['Currículo', 'Vaga']).round(3)
tabela_tfidf = tabela_tfidf[(tabela_tfidf.T != 0).any()]
print('Pesos TF-IDF (termo x documento):')
tabela_tfidf.sort_values('Currículo', ascending=False)

In [ ]:
# Etapa 9 — Similaridade do cosseno (ilustração) entre currículo e vaga
from sklearn.metrics.pairwise import cosine_similarity
sim_cosseno = cosine_similarity(matriz[0:1], matriz[1:2])[0][0]
print(f'Similaridade do cosseno (TF-IDF, ilustrativa): {sim_cosseno:.4f}  ({sim_cosseno:.1%})')

In [ ]:
# Etapa 10 — Extração de habilidades (taxonomia ESCO)
from src.extraction.skills_extractor import ExtratorSkills

extrator = ExtratorSkills()
skills_cv   = set(extrator.extrair_lista(curriculo))
skills_vaga = set(extrator.extrair_lista(vaga))

print('Habilidades no currículo :', sorted(skills_cv))
print('Habilidades na vaga      :', sorted(skills_vaga))
print('Habilidades em comum     :', sorted(skills_cv & skills_vaga))

### Etapa 11 — Score de habilidades e score final (matcher oficial)

Usa `MatcherCurriculoVaga.comparar_um`, que aplica o pré-processamento, o TF-IDF, a similaridade do cosseno e a combinação ponderada **com os pesos reais** definidos no sistema, retornando um objeto `ResultadoMatch`.

In [ ]:
from src.matching.matcher import MatcherCurriculoVaga

matcher = MatcherCurriculoVaga()   # usa os pesos padrão (reais) definidos no __init__
print(f'Pesos do sistema  ->  TF-IDF: {matcher.peso_tfidf}   |   Habilidades: {matcher.peso_skills}')

# Passamos o texto BRUTO: o matcher faz a própria limpeza internamente.
try:
    resultado = matcher.comparar_um(curriculo, vaga, 'vaga_exemplo', 'Desenvolvedor Python')
except TypeError:
    resultado = matcher.comparar_um(curriculo, vaga)   # fallback se a assinatura for só (cv, vaga)

print()
print(f'Score TF-IDF (cosseno)   : {resultado.score_tfidf:.4f}')
print(f'Score de habilidades     : {resultado.score_skills:.4f}')
print(f'Cobertura de habilidades : {resultado.cobertura_skills:.4f}')
print(f'SCORE FINAL              : {resultado.score_final:.4f}  ({resultado.score_percentual}%)')
print(f'Nível de compatibilidade : {resultado.nivel_compatibilidade()}')
print()
print('Habilidades em comum :', resultado.skills_comum)
print('Habilidades ausentes :', resultado.skills_ausentes)
print('Habilidades extras   :', resultado.skills_extras)

In [ ]:
# Tabela-resumo do cálculo do score (copiar para o relatório)
import pandas as pd
resumo = [
    ('Score TF-IDF (cosseno)',   f'{resultado.score_tfidf:.4f}',  f'peso {matcher.peso_tfidf}'),
    ('Score de habilidades',     f'{resultado.score_skills:.4f}', f'peso {matcher.peso_skills}'),
    ('Cobertura de habilidades', f'{resultado.cobertura_skills:.4f}', '-'),
    ('Score final',              f'{resultado.score_final:.4f}',  f'{resultado.score_percentual}%'),
]
pd.DataFrame(resumo, columns=['Componente', 'Valor', 'Observação'])